In [1]:
print("Hello")

Hello


In [3]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
load_dotenv()

True

In [25]:
load_dotenv()
llm = ChatGroq(
    model=os.getenv("GROQ_MODEL"),
    temperature=0.7)

In [39]:
llm.invoke("Hello, how are you today?")

AIMessage(content='<think>\nOkay, the user just greeted me with "Hello, how are you today?" I need to respond appropriately. First, I should acknowledge their greeting and express that I\'m doing well. I want to keep the tone friendly and welcoming.\n\nI should make sure to ask them how they\'re doing in return, showing genuine interest. Maybe add an emoji to keep it warm and approachable. Let me check if there\'s any other nuance I should include. Perhaps a subtle invitation for them to share more if they\'re comfortable. \n\nI need to avoid any formal language and keep it natural. The response should be concise but not too short. Let me put it all together: thank them for asking, state that I\'m doing well, ask about their day, and add a friendly emoji. That should cover it.\n</think>\n\nHello! I\'m doing great, thanks for asking. I\'m an AI, so I don\'t have feelings in the traditional sense, but I\'m always excited to chat and help out. How are you doing today? 😊', additional_kwarg

In [40]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
async def router(user_input: str) -> str:
    """ 
    Identifies what the user's input is and determines the action to be taken.
    """
    routing_prompt_template = ChatPromptTemplate.from_template("""
    You are the **DirectEd Single-Word Intent Router**, a highly constrained and logic-driven system.

    **Your SOLE and ONLY task is to analyze the user's request and return a single, lowercase word corresponding to the correct intent.**

    ### Available Intents (One Word ONLY):

    * **flashcard**: For requests asking for a definition, simple explanation, quick fact, or study aid for memorization.
    * **quiz**: For requests that sound like a quick, informal check of knowledge or self-testing.
    * **assessment**: For requests that require a formal, structured, high-stakes examination (e.g., "exam," "assessment," "formal test").
    * **general_question**: For any informational or open-ended request that does not fit the educational tool categories above (e.g., "Explain X simply," "What is the history of Y?").

    ### Output Constraint (CRITICAL):

    1.  **Return ONLY ONE WORD:** Do not include any punctuation, quotes, spaces, reasoning, or introductory phrases.
    2.  **Case:** The word MUST be lowercase.
    3.  **Source:** The word MUST be one of the four words in the list above: `flashcard`, `quiz`, `assessment`, or `general_question`.

    ### User Request to Analyze:
    {user_input}
    """)
    router_chain = routing_prompt_template | llm | StrOutputParser()

    return await router_chain.ainvoke({
        "user_input": user_input
    })

In [41]:
a = await router("Hello")

In [42]:
a

'<think>\nOkay, let\'s see. The user wrote "Hello". That\'s a greeting. I need to determine the intent based on the available options: flashcard, quiz, assessment, or general_question.\n\nFirst, "flashcard" is for definitions or quick facts. "Quiz" is for self-testing. "Assessment" is formal exams. "General_question" is for open-ended info. The user just said hello, which is a greeting, not a specific request. Since there\'s no actual question or request for any of the educational tools, the best fit is general_question. Wait, but maybe not even that. The instructions say if it doesn\'t fit the others, use general_question. Since the user isn\'t asking for anything specific, but just greeting, maybe it\'s a general question. But maybe there\'s no intent here. Hmm. The system requires one of the four. Since there\'s no clear intent for flashcard, quiz, or assessment, it has to be general_question. Even though the user didn\'t ask a question, the absence of other intent options pushes it

In [43]:
general_chat_llm = ChatGroq(
    model=os.getenv("GROQ_MODEL"),
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.7
)

In [44]:
import logging
import os

load_dotenv()
def logger():
    logging.basicConfig(level=logging.INFO,
                        format='%(asctime)s - %(filename)s - %(levelname)s - %(message)s')
    logger = logging.getLogger(__name__)
    return logger
index_name = os.getenv("PINECONE_INDEX")
logger = logger()

In [46]:
async def route_query(user_request: str) -> str:
    """Route queries intelligently between CrewAI (health) or LangChain (general chat)."""
    router_prompt = ChatPromptTemplate.from_template(
        """
        You are a routing expert. Decide whether to route the user query to 
        'crewai' (for content generation) or 'langchain' (for general chat).
        
        Respond with one word only: crewai or langchain.
        
        User: "{query}"
        Response:
        """
    )
    router_chain = router_prompt | general_chat_llm| StrOutputParser()

    try:
        decision = await router_chain.ainvoke({"query": user_request})
        return decision.strip().lower()
    except Exception as e:
        logger.exception("Router LLM failed. Proceeding with langchain")
        return "langchain"

In [47]:
a = await route_query("Hello, how are you?")

2025-10-10 16:41:32,568 - _client.py - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [49]:
a.title


<function str.title()>